# Forecasting electricity demand

Run **Setup** before presenting. Then follow the slides:
Demand alone → Load the model → The history → Make a forecast → The next day →
A range of possibilities → A shorter history.

The slides introduce our pretrained model: Amazon's Chronos-Bolt Tiny, a
forecasting network with about 9 million parameters. It takes one ordered
history of values and returns nine predicted quantiles for each future step.
See `models/electricity/README.md` for the model's source, input and output.
`demos/electricity_model.py` supplies the loader; `data/electricity/README.md`
attributes the demand data.

## Setup — before class

Install the requirements and run `tools/prepare_electricity_demo.py` once; see
`models/electricity/README.md`. This cell loads the trained model and makes
`Module02` the working directory, so the slide code runs as written.

In [ ]:
from pathlib import Path
import os
import sys

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "demos/electricity_model.py").is_file()
)
sys.path.insert(0, str(ROOT / "demos"))
os.chdir(ROOT)

import numpy as np
import torch
from electricity_inference import plot_forecast
from electricity_model import load_example, load_pipeline

torch.set_num_threads(2)
times = load_example()[0]        # Timestamps, used only to label the plots.
pipeline = load_pipeline()

## Demand alone

Last time's first window had five features. This model takes demand only.

In [ ]:
# Last time's first window: demand, temperature, clear, cloudy, rain.
window = torch.tensor([[[520., 12., 1., 0., 0.],
                        [500., 11., 1., 0., 0.],
                        [480., 10., 0., 1., 0.]]])
toy_history = window[:, :, 0]    # Keep demand at every time position.
print(window.shape, toy_history.shape)
print(toy_history)

## Load the model

Setup already ran this helper. The pipeline wraps the model together with its
input and output processing.

In [ ]:
print(type(pipeline).__name__)

## The history

The file has eight days of hourly demand. The first seven are the history.

In [ ]:
# Load the demand column, not the timestamps.
values = np.loadtxt("data/electricity/neso-hourly.csv",
                    delimiter=",", skiprows=1, usecols=1)
demand = torch.from_numpy(values).to(torch.float32)
history = demand[:168].unsqueeze(0)
print(demand.shape, history.shape)
print(history[0, -1].item())     # MW, Jan 14 at 23:00 UTC

In [ ]:
fig = plot_forecast(times, history)

## Make a forecast

One call returns all 24 hours. Each hour has nine quantiles.

In [ ]:
forecasts = pipeline.predict(history, prediction_length=24)
print(forecasts.shape)          # Axes: history, quantile, future hour.

In [ ]:
print(pipeline.quantiles)

In [ ]:
median = forecasts[0, 4, :]      # First history, median, all 24 hours.
print(median.shape)
print(median[:3].round().tolist())

In [ ]:
# Does the median continue the daily rise and fall?
fig = plot_forecast(times, history, forecasts)

Pause here for **The forecast** in the slides.

## The next day

January 15 was not in the input. Use it now, for comparison only.

In [ ]:
actual = demand[168:]
fig = plot_forecast(times, history, forecasts, actual, days=1)

In [ ]:
print(round(median[0].item()), actual[0].item())
print(round((median[0] - actual[0]).item()))   # MW

## A range of possibilities

Shade between indexes 0 and 8: the 0.1 and 0.9 quantiles.

In [ ]:
fig = plot_forecast(times, history, forecasts, actual, band=True, days=1)

In [ ]:
outside = (actual < forecasts[0, 0]) | (actual > forecasts[0, 8])
print(outside.sum().item(), "of 24 observed hours are outside the band")

About 5 of 24 hours outside the band would be no surprise. These are in a row,
and all above it.

Instructor note: one day is an illustration, not an accuracy test. Whether
these observations were in the model's training data is not established.

## A shorter history

Give the model only the last 24 hours, a Sunday. Does its Monday change?

In [ ]:
short_forecasts = pipeline.predict(history[:, -24:], prediction_length=24)
print("Evening peak from one day: ", round(short_forecasts[0, 4].max().item()), "MW")
print("Evening peak from one week:", round(median.max().item()), "MW")
print("Observed evening peak:     ", round(actual.max().item()), "MW")
fig = plot_forecast(times, history, short_forecasts, actual, days=1)